In [ ]:
import os, json, platform, re, warnings, unicodedata, shutil, html
from datetime import datetime
from typing import List, Tuple, Optional, Any
from pathlib import Path
from collections import Counter

# CPU/Thread/Allocator
os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

import torch
import pandas as pd
import numpy as np
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
transformers.logging.set_verbosity_error()
torch.set_num_threads(12)
torch.set_num_interop_threads(3)

from docx import Document
from striprtf.striprtf import rtf_to_text

# RAG (Chroma)
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# =======================
# CONFIG
# =======================

# 1) Mód generovania
MODE = "title"   # 'title' alebo 'keyword'

# 2) Vstupy/Výstupy
INPUT_DIR   = "./Input"
OUTPUT_DIR  = "./Output"

# 3) Tezaurus/stopwords
THESAURUS_XLSX = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL  = "slovak_terms"
STOPWORDS_TXT  = "./Input/stopword_dictionary.txt"

# 4) Modely a voľba aktívneho (iba jeden sa reálne načíta)
MODELS = {
    "primary": {
        "hf_id": "marcuscedricridia/Mixmix-LlaMAX3.2-1B-Merge",
        "quant4bit": True,
        "dtype": "fp16",
        "gen": dict(max_new_tokens=256, do_sample=True, temperature=0.6,
                    top_p=0.95, repetition_penalty=1.6, top_k=20, min_p=0),
        "batch_size": 4,
        "enable_rag": True,
        "enable_reasoning": True,
    },
    "secondary": {
        "hf_id": "marcuscedricridia/Yell-Qwen2.5-7B-Coder",
        "quant4bit": True,
        "dtype": "fp16",
        "gen": dict(max_new_tokens=256, do_sample=True, temperature=0.6,
                    top_p=0.95, repetition_penalty=1.6, top_k=20, min_p=0),
        "batch_size": 4,
        "enable_rag": True,
        "enable_reasoning": False,
    },
    # "fallback": {...}
}
ACTIVE_MODEL = "secondary"   # 'primary' | 'secondary' | 'fallback'

# 5) Reasoning (self-consistency)
REASONING_SAMPLES = 3
REASONING_TEMP    = 0.6
REASONING_TOP_P   = 0.95

# 6) Text & penalizácie
MAX_TEXT_CHARS = 8000
PENALIZE_CZ = True

# 7) RAG: embedding + store
RAG_PERSIST_DIR     = "./RAG_store"
RAG_COLLECTION      = "sk_legal_local"
# RAG_EMB_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"  # 384-d
RAG_EMB_MODEL_NAME  = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # 768-d
RAG_TOP_K           = 5

# 8) RAG-inspektor (HTML/CSV) + voliteľná 2D projekcia embeddingov
RAG_REPORT = True
RAG_REPORT_DIR = "./Output/rag_report"
RAG_PLOT_PROJECTION = True  # vyžaduje matplotlib a ideálne umap-learn / inak TSNE zo sklearn

# Natvrdo device map
DEVICE_MAP = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"

# =======================
# Stopwords / Tezaurus
# =======================
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, "r", encoding="utf-8") as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f"[STOP] Loaded {len(words)} stopwords")
        return words
    except FileNotFoundError:
        print("[STOP] No stopwords file; continuing.")
        return set()
    except Exception as e:
        print(f"[STOP] Failed to load stopwords: {e}")
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(" - ")[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r"^(.*?)(\s*\(([^)]+)\))\s*$", term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r"[\/,;]", inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", r"\\s+", re.escape(s))
    return s.replace(r"\-", r"[-–—]")

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r"\b" + _wildify(base) + r"(?:\s*\([^)]*\))?\b", re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r"\b" + _wildify(syn) + r"\b", re.IGNORECASE))
    pats_na = [re.compile(r"\b" + _wildify(strip_accents(base)) + r"(?:\s*\([^)]*\))?\b", re.IGNORECASE)]
    pats_na += [re.compile(r"\b" + _wildify(strip_accents(syn)) + r"\b", re.IGNORECASE) for syn in syns]
    return {"canon": canon, "patterns": pats, "patterns_noacc": pats_na, "synonyms": syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"Thesaurus not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r"[,\n;]+", cell))
    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t: continue
        if len(t) == 1 or t.lower() in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    print(f"[TH] Loaded {len(terms)} terms")
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = {(m := build_term_matcher(t))["canon"].lower(): m for t in THESAURUS_TERMS}
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    if not text: return []
    hits, t_na = {}, strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m["patterns"])
        b = sum(len(p.findall(t_na)) for p in m["patterns_noacc"])
        cnt = max(a, b)
        if cnt > 0: hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# =======================
# DOCX/RTF sekcionovanie
# =======================
_HEADING_NAME_PREFIXES = ("heading", "nadpis", "po-heading-id")
_HEADING_EXACT_NAMES   = {"title","subtitle","toc heading","obsah","po-heading-id_"}
_PAGE_LINE_RE = re.compile(r"^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$", re.IGNORECASE)
_RUBRIC_RE    = re.compile(r"([.\-–—]{3,})")

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or "").strip().lower()
    except: name = ""
    try: sid  = (getattr(p.style,"style_id","") or "").lower()
    except: sid = ""
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r"^heading\d+$", name) or re.match(r"^nadpis\s*\d+$", name): return True
    if re.match(r"^heading\d+$", sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, "\n".join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None: buf.append(t)
        if current is not None and buf:
            sections.append((current, "\n".join(buf).strip()))
        return [("__DOCUMENT__", "\n".join(full).strip())] + sections
    except Exception as e:
        print(f"[WARN] DOCX split failed {path}: {e}")
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r"^\d+(\.\d+)*\s+\S", l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,"r",encoding="utf-8").read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, "\n".join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None: buf.append(ln)
        if current is not None and buf:
            sections.append((current, "\n".join(buf).strip()))
        return [("__DOCUMENT__", "\n".join(full).strip())] + sections
    except Exception as e:
        print(f"[WARN] RTF split failed {path}: {e}")
        return []

# =======================
# LLM Loader (1 vybraný model)
# =======================
def _bitsandbytes_qconf():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

def load_selected_llm(active: str):
    cfg = MODELS[active]
    hf_id = cfg["hf_id"]
    qconf = _bitsandbytes_qconf() if cfg["quant4bit"] else None
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconf,
        torch_dtype=None if qconf is not None else torch.float16
    )
    pipe = pipeline(
        "text-generation",
        model=model, tokenizer=tok, device_map=DEVICE_MAP,
        return_full_text=False, **cfg["gen"]
    )
    return pipe, f"{active}:{hf_id}"

LLM_PIPE, USED_MODEL = load_selected_llm(ACTIVE_MODEL)
print(f"[LLM] Loaded {USED_MODEL}")

# =======================
# RAG init (iba ak povolené)
# =======================
def _ensure_chroma_vectorstore() -> Optional[Chroma]:
    if not MODELS[ACTIVE_MODEL]["enable_rag"]:
        print("[RAG] Disabled.")
        return None
    os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)

    def _open_store():
        return Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=embeddings,
        )

    try:
        vs = _open_store()
        _ = vs._collection.get(limit=1)
        print(f"[RAG] Enabled (top_k={RAG_TOP_K}).")
        return vs
    except Exception as e:
        print(f"[RAG] Vectorstore init failed, recreating: {e}")
        if os.path.isdir(RAG_PERSIST_DIR):
            shutil.rmtree(RAG_PERSIST_DIR, ignore_errors=True)
            os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
        vs = _open_store()
        print("[RAG] Vectorstore ready (fresh).")
        return vs

VECTORSTORE = _ensure_chroma_vectorstore()

def rag_upsert(doc_id: str, texts: List[str], metas: List[dict]):
    if VECTORSTORE is None or not texts: return
    try:
        VECTORSTORE.add_texts(texts=texts, metadatas=metas, ids=[f"{doc_id}_{i}" for i in range(len(texts))])
    except Exception as e:
        print(f"[RAG] upsert failed: {e}")

def rag_retrieve_with_scores(query: str, top_k: int = RAG_TOP_K):
    if VECTORSTORE is None: return []
    try:
        docs = VECTORSTORE.similarity_search_with_score(query, k=top_k)
        return [(d.page_content, float(s)) for d, s in docs]
    except Exception as e:
        print(f"[RAG] retrieve_with_scores failed: {e}")
        return []

# =======================
# RAG-inspektor (vizualizácia a metriky)
# =======================
def _html_highlight_terms(text: str, terms: List[str]) -> str:
    if not text: return ""
    safe = html.escape(text)
    for t in terms[:40]:
        if not t: continue
        pat = re.compile(rf"(?i)\b{re.escape(t)}\b")
        safe = pat.sub(lambda m: f"<mark>{m.group(0)}</mark>", safe)
    return safe

def _quality_metrics(retrieved: List[Tuple[str,float]], prio_terms: List[str]) -> dict:
    scores = [s for _, s in retrieved] or [0.0]
    all_text = " \n ".join([t for t,_ in retrieved]).lower()
    found = set()
    for t in prio_terms[:40]:
        if re.search(rf"(?i)\b{re.escape(t)}\b", all_text):
            found.add(t)
    cz_pen = 1 if re.search(all_text) else 0
    return dict(
        topk=len(retrieved),
        score_avg=float(np.mean(scores)),
        score_med=float(np.median(scores)),
        hit_rate=1.0 if len(found)>0 else 0.0,
        coverage=len(found)/max(1,len(prio_terms[:40])),
        cz_penalty=cz_pen,
        ctx_chars=len(all_text)
    )

def rag_save_case_report(file_name: str, header: str, query_text: str,
                         prio_terms: List[str], retrieved: List[Tuple[str,float]],
                         out_dir: str) -> str:
    from plotly import graph_objects as go
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    labels = [f"ctx{i+1}" for i in range(len(retrieved))]
    scores = [round(s,4) for _,s in retrieved]
    fig = go.Figure(data=[go.Bar(x=labels, y=scores)])
    fig.update_layout(title="RAG similarity (cosine / inner-product)",
                      xaxis_title="Pasáž", yaxis_title="Score")
    rows_html = []
    for i,(txt,sc) in enumerate(retrieved,1):
        rows_html.append(
            f"<tr><td><b>ctx{i}</b></td><td>{sc:.4f}</td>"
            f"<td style='line-height:1.5'>{_html_highlight_terms(txt, prio_terms)}</td></tr>"
        )
    prio_html = ", ".join([html.escape(t) for t in prio_terms[:20]]) or "(žiadne)"
    table_html = (
        "<table border='1' cellspacing='0' cellpadding='6'>"
        "<tr><th>Id</th><th>Score</th><th>Text</th></tr>"
        + "\n".join(rows_html) + "</table>"
    )
    plot_div = fig.to_html(full_html=False, include_plotlyjs="cdn")
    html_doc = f"""
                <!doctype html>
                <html><head><meta charset="utf-8"><title>RAG report</title>
                <style>body{{font-family:system-ui,Segoe UI,Roboto,Arial;margin:24px;}}
                mark{{background:#ffe08a;padding:0 2px;border-radius:2px}}
                pre{{white-space:pre-wrap}}</style></head>
                <body>
                <h2>RAG Report</h2>
                <p><b>Súbor:</b> {html.escape(file_name)}<br/>
                <b>Sekcia:</b> {html.escape(header)}</p>
                <h3>Query</h3>
                <pre>{html.escape(query_text)}</pre>
                <p><b>PRIO termíny:</b> {prio_html}</p>
                <h3>Skóre</h3>
                {plot_div}
                <h3>Top-K pasáže</h3>
                {table_html}
                </body></html>
                """
    safe_header = re.sub(r"[^a-zA-Z0-9]+","_", header)[:60]
    out_path = os.path.join(out_dir, f"{Path(file_name).stem}__{safe_header}.html")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(html_doc)
    return out_path

def rag_projection_png(vs: Chroma, out_dir: str, fname: str="projection.png", max_points: int=1000):
    if not RAG_PLOT_PROJECTION: return None
    try:
        import matplotlib.pyplot as plt
        X = vs._collection.get(include=["embeddings"], limit=max_points)
        embs = np.array(X.get("embeddings", []))
        if embs is None or len(embs) < 10: return None
        try:
            import umap
            Z = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine").fit_transform(embs)
        except Exception:
            from sklearn.manifold import TSNE
            Z = TSNE(perplexity=30, init="random", learning_rate="auto").fit_transform(embs)
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        plt.figure(figsize=(6,5))
        plt.scatter(Z[:,0], Z[:,1], s=6)
        path = os.path.join(out_dir, fname)
        plt.savefig(path, dpi=140, bbox_inches="tight"); plt.close()
        return path
    except Exception as e:
        print(f"[RAG] projection failed: {e}")
        return None

# =======================
# PROMPTS & Reasoning
# =======================
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars: return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def _reasoning_wrap(task_prompt: str) -> str:
    return (
        "Premýšľaj potichu a dôkladne, ale do odpovede prosím nevypisuj postup ani úvahy.\n"
        "Vráť len finálny výsledok presne v požadovanom formáte.\n\n" + task_prompt
    )

def prompt_title(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    rag  = ("\n\nDOPLNKOVÝ KONTEXT (RAG):\n" + "\n---\n".join(rag_ctx)) if rag_ctx else ""
    core = (
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný slovenský právny nadpis (3–6 slov) k textu nižšie.\n"
        "Požiadavky: vecný, bez bodky na konci, žiadne úvodzovky. Preferuj slovenské výrazy.\n"
        "Vráť len samotný nadpis, nič iné.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}{rag}\n\n"
        "NADPIS:"
    )
    cfg = MODELS[ACTIVE_MODEL]
    return _reasoning_wrap(core) if cfg["enable_reasoning"] and REASONING_SAMPLES >= 2 else core

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    rag  = ("\n\nDOPLNKOVÝ KONTEXT (RAG):\n" + "\n---\n".join(rag_ctx)) if rag_ctx else ""
    core = (
        "ÚLOHA: Vyber JEDEN najvhodnejší slovenský právny pojem (1–4 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky. Vyhni sa češtine.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}{rag}\n\n"
        "POJEM:"
    )
    cfg = MODELS[ACTIVE_MODEL]
    return _reasoning_wrap(core) if cfg["enable_reasoning"] and REASONING_SAMPLES >= 2 else core

def _normalize_output_item(item) -> str:
    raw = ""
    if item is None:
        raw = ""
    elif isinstance(item, dict):
        raw = item.get("generated_text") or item.get("text") or item.get("summary_text") or ""
    elif isinstance(item, list):
        if item and isinstance(item[0], dict):
            raw = item[0].get("generated_text") or item[0].get("text") or item[0].get("summary_text") or ""
        else:
            raw = str(item)
    else:
        raw = str(item)
    txt = raw or ""
    txt = re.sub(r'^[\-\•\:\s]+', '', txt).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    lines = [ln for ln in txt.splitlines() if ln.strip()]
    return lines[0].strip() if lines else ""

def _shape_group(flat_out, n_inputs: int, nrs: int):
    blob = []
    for item in flat_out:
        if isinstance(item, list) and item and isinstance(item[0], dict):
            blob.extend(item)
        else:
            blob.append(item)
    grouped = []
    it = iter(blob)
    for _ in range(n_inputs):
        chunk = []
        for _ in range(nrs):
            try:    chunk.append(next(it))
            except StopIteration: chunk.append(None)
        grouped.append(chunk)
    return grouped

def safe_llm_call_batch(
    pipe, inputs, *, batch_size=1, num_return_sequences=1,
    max_new_tokens=None, temperature=None, top_p=None,
    max_retries=2, min_batch=1, min_new_tokens=32, min_samples=1, sleep_s=0.0
):
    import time
    bs  = max(1, int(batch_size))
    nrs = max(1, int(num_return_sequences))
    mnt = max_new_tokens

    def _call(_bs, _nrs, _mnt):
        kwargs = dict(batch_size=_bs, num_return_sequences=_nrs)
        if _mnt is not None: kwargs["max_new_tokens"] = _mnt
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None: kwargs["top_p"] = top_p
        return pipe(inputs, **kwargs)

    for _ in range(max_retries + 1):
        try:
            out = _call(bs, nrs, mnt)
            return out, bs, nrs, mnt
        except RuntimeError as e:
            msg = str(e)
            if ("CUDA out of memory" in msg) or ("device-side assert" in msg):
                changed = False
                if bs > min_batch: bs = max(min_batch, bs // 2); changed = True
                elif nrs > min_samples: nrs = max(min_samples, nrs // 2); changed = True
                elif mnt and mnt > min_new_tokens: mnt = max(min_new_tokens, mnt // 2); changed = True
                if not changed: break
                try:
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats()
                except Exception:
                    pass
                time.sleep(sleep_s)
            else:
                break

    # per-input fallback
    outputs, real_mnt = [], max(16, min(64, (min_new_tokens if min_new_tokens else (mnt or 32))))
    for prompt in inputs:
        try:
            out = pipe(prompt, batch_size=1, num_return_sequences=1, max_new_tokens=real_mnt)
            outputs.append(out[0] if isinstance(out, list) else out)
        except Exception:
            outputs.append("")
    return outputs, 1, 1, real_mnt

def batched_generate_texts(prompts: List[str], batch_size: int, gen_cfg: dict) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, _, _, _ = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=1,
            max_new_tokens=gen_cfg.get("max_new_tokens", 96),
            temperature=gen_cfg.get("temperature"),
            top_p=gen_cfg.get("top_p")
        )
        for item in out:
            outputs.append(_normalize_output_item(item))
    if len(outputs) != len(prompts):
        print(f"[WARN] single: outputs={len(outputs)} != inputs={len(prompts)}")
        while len(outputs) < len(prompts): outputs.append("")
        if len(outputs) > len(prompts): outputs = outputs[:len(prompts)]
    return outputs

def batched_generate_texts_reasoned(jobs, batch_size: int, gen_cfg: dict) -> List[str]:
    prompts = [j[-3] for j in jobs]  # index where prompt stored in job tuple
    outputs = []
    nrs = max(1, min(REASONING_SAMPLES, 6))
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, _, _, _ = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=nrs,
            max_new_tokens=min(gen_cfg.get("max_new_tokens", 96), 128),
            temperature=REASONING_TEMP,
            top_p=REASONING_TOP_P,
            max_retries=2, min_batch=1, min_new_tokens=32, min_samples=1
        )
        grouped = _shape_group(out, n_inputs=len(batch), nrs=nrs)
        for idx_in_batch, cand_list in enumerate(grouped):
            mode, f, header, prio_terms, short_text, _prompt, _retrieved_scored = jobs[i + idx_in_batch]
            texts = [_normalize_output_item(c) for c in cand_list if c is not None]
            if not texts:
                outputs.append(""); continue
            def score_slovak(t: str) -> float:
                if not t: return -1e9
                cz_pen = 0.3 if re.search(r'\b(př|ř|ě|í|ý|ů|č|ď|ť|ň)\b', t.lower()) else 0.0
                base = max(0.0, 1.0 - 0.01 * max(0, len(t) - 60))
                return base - cz_pen
            best = max(texts, key=score_slovak)
            outputs.append(best)

    if len(outputs) != len(prompts):
        print(f"[WARN] reasoned: outputs={len(outputs)} != inputs={len(prompts)}")
        while len(outputs) < len(prompts): outputs.append("")
        if len(outputs) > len(prompts): outputs = outputs[:len(prompts)]
    return outputs

def penalize_czech(s: str) -> str:
    if not PENALIZE_CZ or not s: return s
    s2 = s
    s2 = re.sub(r"\břízení\b", "konanie", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bustanovení\b", "ustanovenie", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bsprávní\b", "správne", s2, flags=re.IGNORECASE)
    return s2

# =======================
# DRIVER
# =======================
def process_all(input_dir=INPUT_DIR):
    cfg = MODELS[ACTIVE_MODEL]
    rows, files = [], sorted(os.listdir(input_dir), key=str.lower)

    rag_active = bool(VECTORSTORE is not None)
    reasoning_active = bool(cfg["enable_reasoning"] and REASONING_SAMPLES >= 2)

    # 1) Build jobs + (voliteľný) RAG upsert
    jobs = []  # (mode, file, header, prio_terms, short_text, prompt, retrieved_scored)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f"No text in {f}"); continue

        # RAG upsert (iba sekcie, nie __DOCUMENT__)
        if rag_active:
            section_texts, metas = [], []
            for idx, (header, text) in enumerate(secs):
                if header == "__DOCUMENT__": continue
                if text.strip():
                    section_texts.append(text.strip())
                    metas.append({"file": f, "header": header, "idx": idx})
            if section_texts:
                rag_upsert(os.path.splitext(f)[0], section_texts, metas)

        for header, text in secs:
            if not text.strip(): continue
            short_text = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]
            retrieved_scored = rag_retrieve_with_scores(short_text, top_k=RAG_TOP_K) if rag_active else []
            rag_ctx = [t for (t,_) in retrieved_scored] if rag_active else []

            if MODE == "title":
                prompt = prompt_title(short_text, prio_terms, sample_terms, rag_ctx)
                jobs.append(("title", f, header, prio_terms, short_text, prompt, retrieved_scored))
            else:
                prompt = prompt_keyword(short_text, prio_terms, sample_terms, rag_ctx)
                jobs.append(("keyword", f, header, prio_terms, short_text, prompt, retrieved_scored))
        print(f"Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("No jobs queued.")
        return pd.DataFrame([])

    # 2) Inference
    if reasoning_active:
        gens = batched_generate_texts_reasoned(jobs, batch_size=cfg["batch_size"], gen_cfg=cfg["gen"])
    else:
        prompts = [j[-3] for j in jobs]  # prompt field
        gens = batched_generate_texts(prompts, batch_size=cfg["batch_size"], gen_cfg=cfg["gen"])

    # 3) Kompozícia + jemná CZ->SK korekcia + RAG report
    report_rows = []
    proj_path = rag_projection_png(VECTORSTORE, RAG_REPORT_DIR) if (RAG_REPORT and rag_active) else None

    for (mode, f, header, prio_terms, short_text, prompt, retrieved_scored), gen in zip(jobs, gens):
        gen = penalize_czech(gen)

        # Riadok do hlavného výsledku
        if mode == "title":
            rows.append({
                "file": f, "header": header,
                "suggested_title": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "file": f, "header": header,
                "top_keyword": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })

        # RAG report/metriky
        if RAG_REPORT and rag_active:
            met = _quality_metrics(retrieved_scored, prio_terms)
            html_path = rag_save_case_report(f, header, short_text, prio_terms, retrieved_scored, RAG_REPORT_DIR)
            report_rows.append({ "file": f, "header": header, "report_html": html_path, **met })

    # 4) Export
    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == "title":
        df = pd.DataFrame(rows, columns=["file","header","suggested_title","method","priority_terms"])
        stub = "titles"
    else:
        df = pd.DataFrame(rows, columns=["file","header","top_keyword","method","priority_terms"])
        stub = "keywords"

    csv_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")

    if RAG_REPORT and rag_active and report_rows:
        Path(RAG_REPORT_DIR).mkdir(parents=True, exist_ok=True)
        qual_csv = os.path.join(RAG_REPORT_DIR, f"rag_quality_{today}.csv")
        pd.DataFrame(report_rows).to_csv(qual_csv, index=False, encoding="utf-8-sig")
        print(f"[RAG] Quality CSV + HTML reporty -> {RAG_REPORT_DIR}")
        if proj_path:
            print(f"[RAG] 2D projekcia embeddingov -> {proj_path}")

    return df

# =======================
# MAIN
# =======================
if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=r".*Some weights.*not of the same dtype.*")
    warnings.filterwarnings("ignore", message=r".*Tokenizer parallelism.*")

    df = process_all(INPUT_DIR)
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))

[STOP] Loaded 76 stopwords
[TH] Loaded 3000 terms
[LLM] Loaded primary:marcuscedricridia/Mixmix-LlaMAX3.2-1B-Merge


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_19452\1314732541.py:317: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)
C:\Users\nyx3t\AppData\Local\Temp\ipykernel_19452\1314732541.py:320: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


[RAG] Enabled (top_k=5).
Queued 117694.docx with 8 sections.
Queued 117695.docx with 2 sections.
Queued 117696.docx with 11 sections.
Queued 117697.docx with 15 sections.
Queued 117698.docx with 8 sections.
Queued 117699.docx with 18 sections.
Queued 117700.docx with 1 sections.
Queued 117701.docx with 6 sections.
Queued 117702.docx with 12 sections.
Queued 117703.docx with 10 sections.


KeyboardInterrupt: 